# BlindTape 核心训练引擎 Demo

对应技术方案「开发顺序」第2步：随机抽样引擎 + 不带UI的训练循环，先在这里跑通，
后面再挪到网页前端。**代码和真实日期在训练过程中不会显示**，只有跑到最后的"揭晓"cell才会显示。

用法：
1. 先跑「初始化」部分的几个cell，抽到一个随机窗口。
2. 反复交替运行"推进一天"和"下操作"两个cell，直到你觉得该收手了（或者窗口自然走完）。
3. 跑「训练结束」部分看统计数据。
4. 跑「揭晓」部分看真实标的和日期。
5. 跑「导出」把这次训练存成JSON。


## 初始化

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False and Path.cwd().name == "notebooks" else Path.cwd()
if (PROJECT_ROOT / "data_layer").exists() is False:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 让中文能正常显示在图上（macOS自带PingFang SC；其他系统找不到就退回默认字体，
# 图还是能画，只是中文label会变方块，不影响功能）
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

from data_layer import db, pool
from data_layer.config import load_config
from training_engine.sampling import sample_window
from training_engine.engine import TrainingSession
from training_engine import export as export_mod

cfg = load_config(str(PROJECT_ROOT / "config.yaml"))
print("db_path:", cfg.db_path, "| window_trading_days:", cfg.sampling.window_trading_days)


In [ ]:
# 标的池：优先复用已经跑过 build-pool 生成的CSV，没有就现算一次
pool_csv = PROJECT_ROOT / cfg.output_pool_csv
if pool_csv.exists():
    # dtype强制指定symbol为str：纯数字的ETF代码列不指定的话会被pandas推断成int64
    pool_df = pd.read_csv(pool_csv, dtype={"symbol": str})
else:
    with db.connect(str(PROJECT_ROOT / cfg.db_path)) as _conn:
        pool_df = pool.build_pool(_conn, cfg.pool_filter)

print("标的池总数:", len(pool_df), "| 入池:", int(pool_df["in_pool"].sum()))


In [ ]:
def plot_candles(ohlcv: pd.DataFrame, title: str = "", x_labels=None):
    """手工画K线，不依赖真实日期（训练过程中用这个，行情看得到、日期看不到）。"""
    fig, ax = plt.subplots(figsize=(11, 4))
    for i, row in ohlcv.reset_index(drop=True).iterrows():
        up = row["close"] >= row["open"]
        color = "red" if up else "green"  # A股习惯：红涨绿跌
        ax.plot([i, i], [row["low"], row["high"]], color=color, linewidth=1, zorder=1)
        bottom = min(row["open"], row["close"])
        height = abs(row["close"] - row["open"])
        ax.add_patch(mpatches.Rectangle((i - 0.3, bottom), 0.6, height or 1e-6, color=color, zorder=2))
    ax.set_xlim(-1, len(ohlcv))
    ax.set_title(title)
    if x_labels is not None:
        step = max(1, len(x_labels) // 12)
        ticks = list(range(0, len(x_labels), step))
        ax.set_xticks(ticks)
        ax.set_xticklabels([x_labels[t] for t in ticks], rotation=45, ha="right")
    else:
        ax.set_xlabel("第几天（训练内部序号，不代表真实日期）")
    fig.tight_layout()
    plt.show()


In [ ]:
seed = random.randrange(10**9)  # 记下这个seed就能复现同一次抽样
rng = random.Random(seed)

with db.connect(str(PROJECT_ROOT / cfg.db_path)) as _conn:
    sample = sample_window(_conn, pool_df, cfg.sampling, rng=rng)

session = TrainingSession(sample.ohlcv)
print(f"抽到一个 {session.total_bars} 天的窗口（代码和日期已隐藏，seed={seed}）")


## 训练循环

反复交替运行下面两个cell：
- 「推进一天」：看到新的一根K线
- 「下操作」：如果想买/加/卖/清仓就改好参数再运行；不想操作就跳过，直接回去跑「推进一天」

action 可选：`"buy"`（开仓）/ `"add"`（加仓）/ `"sell"`（减仓）/ `"liquidate"`（清仓）。
size 是仓位变动比例（0~1 之间，比如0.3代表本次动用30%虚拟资金的仓位），liquidate时会被忽略。


In [ ]:
bar = session.reveal_next()
if bar is None:
    print("窗口已经走完，没有更多K线了，去跑「训练结束」部分")
else:
    plot_candles(session.ohlcv.iloc[: session.cursor + 1],
                 title=f"第 {session.cursor + 1}/{session.total_bars} 天 | 当前仓位 {session.position:.0%}")
    print(bar)


In [ ]:
action = "buy"   # "buy" / "add" / "sell" / "liquidate"
size = 0.3       # 仓位变动比例，0~1

record = session.act(action, size)
print(record)


## 训练结束

In [ ]:
session.finish()
stats = session.stats()
print(stats)

session.equity_curve().plot(title="权益曲线（净值，起点=1.0）", figsize=(11, 3))
plt.show()


## 揭晓

In [ ]:
print(f"真实标的：{sample.symbol} {sample.name}")
print(f"真实区间：{sample.start_date} ~ {sample.end_date}")

dates = sample.ohlcv["trade_date"].tolist()
plot_candles(sample.ohlcv, title=f"{sample.symbol} {sample.name}  {sample.start_date}~{sample.end_date}", x_labels=dates)

# 在图上标出你的操作点
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(sample.ohlcv["close"].tolist(), color="gray")
for a in session.actions:
    color = {"buy": "red", "add": "orange", "sell": "blue", "liquidate": "black"}[a.action]
    ax.scatter([a.day_index], [a.price], color=color, zorder=3, label=a.action)
    ax.annotate(a.action, (a.day_index, a.price))
ax.set_title("操作点位（相对收盘价）")
plt.show()


## 导出

In [ ]:
export_data = export_mod.build_export(session, sample, reveal=True)
out_path = PROJECT_ROOT / "data" / f"session_{sample.symbol}_{sample.start_date}.json"
export_mod.save_json(export_data, str(out_path))
print("已导出:", out_path)
export_data["performance"]
